In [0]:
%run /Shared/insclm_capstone/NB_00_config_loader.py

[SecretScope(name=' kv-insclm-cap-11'), SecretScope(name='kv-insclm')]

[SecretMetadata(key='adls-abfss-base'),
 SecretMetadata(key='adls-account-key'),
 SecretMetadata(key='adls-account-name'),
 SecretMetadata(key='adls-audit-path'),
 SecretMetadata(key='adls-base-url'),
 SecretMetadata(key='adls-bronze-path'),
 SecretMetadata(key='adls-container-name'),
 SecretMetadata(key='adls-gold-path'),
 SecretMetadata(key='adls-raw-path'),
 SecretMetadata(key='adls-rejected-path'),
 SecretMetadata(key='adls-silver-path'),
 SecretMetadata(key='database-workspace-url'),
 SecretMetadata(key='databricks-cluster-id'),
 SecretMetadata(key='databricks-pat'),
 SecretMetadata(key='file-claim-status-updates'),
 SecretMetadata(key='file-claims'),
 SecretMetadata(key='file-customer-master'),
 SecretMetadata(key='file-policy-master'),
 SecretMetadata(key='github-pat'),
 SecretMetadata(key='github-repo-url'),
 SecretMetadata(key='sql-admin-name'),
 SecretMetadata(key='sql-admin-password'),
 SecretMetadata(key='sql-connection-string'),
 SecretMetadata(key='sql-database-name'),
 S

✅ Config loaded from Key Vault successfully.
   ADLS Account  : [REDACTED]
   Container     : [REDACTED]
   ABFSS Base    : [REDACTED]
   RAW path      : [REDACTED][REDACTED]
   BRONZE path   : [REDACTED][REDACTED]
   SILVER path   : [REDACTED][REDACTED]
   GOLD path     : [REDACTED][REDACTED]
   REJECTED path : [REDACTED][REDACTED]
   AUDIT path    : [REDACTED][REDACTED]
   SQL Server    : [REDACTED]
   SQL Database  : [REDACTED]


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from datetime import datetime

# Unity Catalog — NO LOCATION
spark.sql("CREATE DATABASE IF NOT EXISTS silver_insclm")
spark.sql("CREATE DATABASE IF NOT EXISTS rejected_insclm")
print("✅ Databases ready")

✅ Databases ready


In [0]:
print("\n📥 Building silver_customer_dim...")
bronze_customer = spark.table(
    "bronze_insclm.bronze_customer_master")
print(f"   Bronze rows: {bronze_customer.count():,}")

silver_customer = (bronze_customer
    .withColumn("dob_parsed",
        F.coalesce(
            F.to_date(F.col("dob"), "yyyy-MM-dd HH:mm:ss.SSSSSSS"),
            F.to_date(F.col("dob"), "yyyy-MM-dd HH:mm:ss"),
            F.to_date(F.col("dob"), "yyyy-MM-dd"),
            F.to_date(F.col("dob"), "dd-MM-yyyy"),
            F.to_date(F.col("dob"), "MM/dd/yyyy")
        ))
    .withColumn("age_years",
        F.when(F.col("dob_parsed").isNotNull(),
            F.floor(F.months_between(
                F.current_date(),
                F.col("dob_parsed")) / 12))
        .otherwise(F.lit(None)))
    .withColumn("email_valid",
        F.col("email").rlike(
            r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"))
    .withColumn("phone_valid",
        F.length(F.col("phone")) == 10)
    .withColumn("_silver_loaded_at", F.current_timestamp())
    .drop("dob")
    .withColumnRenamed("dob_parsed", "dob")
    .dropDuplicates(["customer_id"]))

silver_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_insclm.silver_customer_dim")

c = spark.table("silver_insclm.silver_customer_dim").count()
print(f"✅ silver_customer_dim → {c:,} rows  (expected 1,000)")

null_dob = spark.table("silver_insclm.silver_customer_dim") \
    .filter(F.col("dob").isNull()).count()
print(f"   NULL dob count : {null_dob}  (expected 0)")

print("\nSample dob + age:")
spark.table("silver_insclm.silver_customer_dim") \
    .select("customer_id", "dob", "age_years") \
    .show(5, truncate=False)


📥 Building silver_customer_dim...
   Bronze rows: 1,000
✅ silver_customer_dim → 1,000 rows  (expected 1,000)
   NULL dob count : 0  (expected 0)

Sample dob + age:
+-----------+----------+---------+
|customer_id|dob       |age_years|
+-----------+----------+---------+
|ICUST00275 |1997-01-22|29       |
|ICUST00469 |1976-09-06|49       |
|ICUST00531 |1963-06-13|62       |
|ICUST00931 |1964-01-16|62       |
|ICUST00514 |1999-10-18|26       |
+-----------+----------+---------+
only showing top 5 rows


In [0]:
# Rejection logic — no dependencies
print("\n📥 Applying rejection logic on claims...")
bronze_claims = spark.table("bronze_insclm.bronze_claims")
print(f"   Bronze claims: {bronze_claims.count():,}")

rejected_conditions = (
    F.col("claim_id").isNull()                       |
    F.col("policy_id").isNull()                      |
    F.col("customer_id").isNull()                    |
    F.col("claim_amount").isNull()                   |
    (F.col("claim_amount") <= 0)                     |
    F.col("document_status").isNull()                |
    (F.col("document_status") == "Missing")          |
    (F.col("claim_date") > F.current_date())
)

rejected_claims = (bronze_claims
    .filter(rejected_conditions)
    .withColumn("rejection_reason",
        F.when(F.col("claim_id").isNull(),
               "MISSING_CLAIM_ID")
        .when(F.col("policy_id").isNull(),
               "ORPHAN_POLICY_ID")
        .when(F.col("customer_id").isNull(),
               "ORPHAN_CUSTOMER_ID")
        .when(F.col("claim_amount").isNull(),
               "MISSING_CLAIM_AMOUNT")
        .when(F.col("claim_amount") <= 0,
               "NEGATIVE_CLAIM_AMOUNT")
        .when(F.col("document_status") == "Missing",
               "MISSING_DOCUMENT_STATUS")
        .when(F.col("claim_date") > F.current_date(),
               "FUTURE_CLAIM_DATE")
        .otherwise("UNKNOWN"))
    .withColumn("rejected_at", F.current_timestamp()))

good_claims = bronze_claims.filter(~rejected_conditions)

rejected_claims.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("rejected_insclm.rejected_claims")

print(f"✅ rejected_claims → {rejected_claims.count():,}  (expected ~120)")
print(f"✅ good_claims     → {good_claims.count():,}  (expected ~2,080)")

print("\nRejection breakdown:")
rejected_claims.groupBy("rejection_reason") \
    .count().orderBy(F.desc("count")).show(truncate=False)

print("\n" + "=" * 55)
print("⏳ STOP HERE")
print("   Message Sreya: Bronze is ready run NB_04")
print("   Wait for her to confirm policy_dim is ready")
print("   Then run Cells 5, 6, 7")
print("=" * 55)


📥 Applying rejection logic on claims...
   Bronze claims: 2,200
✅ rejected_claims → 120  (expected ~120)
✅ good_claims     → 2,080  (expected ~2,080)

Rejection breakdown:
+-----------------------+-----+
|rejection_reason       |count|
+-----------------------+-----+
|MISSING_DOCUMENT_STATUS|120  |
+-----------------------+-----+


⏳ STOP HERE
   Message Sreya: Bronze is ready run NB_04
   Wait for her to confirm policy_dim is ready
   Then run Cells 5, 6, 7


In [0]:
print("\n📥 Loading silver_policy_dim and customer_dim...")
policy_dim   = spark.table("silver_insclm.silver_policy_dim")
customer_dim = spark.table("silver_insclm.silver_customer_dim")

print(f"   policy_dim   : {policy_dim.count():,}")
print(f"   customer_dim : {customer_dim.count():,}")

# Join using current policy version only
claims_with_policy = (good_claims.alias("c")
    .join(
        policy_dim.filter(
            F.col("is_current") == True).alias("p"),
        F.col("c.policy_id") == F.col("p.policy_id"),
        "left"
    ))

claims_with_all = (claims_with_policy
    .join(customer_dim.alias("cu"),
          F.col("c.customer_id") == F.col("cu.customer_id"),
          "left"))

print(f"✅ Joined rows: {claims_with_all.count():,}")


📥 Loading silver_policy_dim and customer_dim...
   policy_dim   : 1,500
   customer_dim : 1,000
✅ Joined rows: 2,080


In [0]:
customer_window = Window.partitionBy("c.customer_id")

silver_claims_fact = (claims_with_all
    .withColumn("customer_claim_count",
        F.count("c.claim_id").over(customer_window))
    .withColumn("amount_exceeds_coverage_flag",
        F.when(F.col("c.claim_amount") >
               F.col("p.coverage_amount"), True)
        .otherwise(False))
    .withColumn("inactive_policy_flag",
        F.col("p.policy_status")
        .isin("Cancelled","Lapsed","Expired"))
    .withColumn("incomplete_docs_flag",
        F.col("c.document_status")
        .isin("Incomplete","Missing"))
    .withColumn("high_frequency_flag",
        F.col("customer_claim_count") > 5)
    .withColumn("suspicious_score",
        F.col("amount_exceeds_coverage_flag").cast("int") +
        F.col("inactive_policy_flag").cast("int") +
        F.col("incomplete_docs_flag").cast("int") +
        F.col("high_frequency_flag").cast("int"))
    .withColumn("_silver_loaded_at", F.current_timestamp())
    .select(
        F.col("c.claim_id"),
        F.col("c.policy_id"),
        F.col("p.policy_sk"),
        F.col("c.customer_id"),
        F.col("cu.customer_name"),
        F.col("cu.city"),
        F.col("cu.state"),
        F.col("cu.risk_category"),
        F.col("p.policy_type"),
        F.col("p.coverage_amount"),
        F.col("p.premium_amount"),
        F.col("p.policy_status").alias("policy_status_at_claim"),
        F.col("c.claim_date"),
        F.col("c.claim_amount"),
        F.col("c.claim_reason"),
        F.col("c.document_status"),
        "amount_exceeds_coverage_flag",
        "inactive_policy_flag",
        "incomplete_docs_flag",
        "high_frequency_flag",
        "customer_claim_count",
        "suspicious_score",
        F.col("c.ingestion_date"),
        "_silver_loaded_at"
    ))

print("✅ silver_claims_fact built")
print(f"   Rows: {silver_claims_fact.count():,}")

✅ silver_claims_fact built
   Rows: 2,080


In [0]:
silver_claims_fact.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_insclm.silver_claims_fact")

count = spark.table("silver_insclm.silver_claims_fact").count()
print(f"✅ silver_claims_fact → {count:,} rows  (expected 2,080)")

print("\n── Business Flag Summary ─────────────────────────────")
spark.table("silver_insclm.silver_claims_fact").agg(
    F.count("claim_id").alias("total_claims"),
    F.sum(F.col("amount_exceeds_coverage_flag").cast("int"))
     .alias("exceeds_coverage"),
    F.sum(F.col("inactive_policy_flag").cast("int"))
     .alias("inactive_policy"),
    F.sum(F.col("incomplete_docs_flag").cast("int"))
     .alias("incomplete_docs"),
    F.sum(F.col("high_frequency_flag").cast("int"))
     .alias("high_frequency"),
).show()

print("✅ NB_02 complete!")
print("   Next: Run NB_03")

✅ silver_claims_fact → 2,080 rows  (expected 2,080)

── Business Flag Summary ─────────────────────────────
+------------+----------------+---------------+---------------+--------------+
|total_claims|exceeds_coverage|inactive_policy|incomplete_docs|high_frequency|
+------------+----------------+---------------+---------------+--------------+
|        2080|               0|            447|            524|           672|
+------------+----------------+---------------+---------------+--------------+

✅ NB_02 complete!
   Next: Run NB_03
